# TN2211 Session 12:  Wave propagation in electrical circuits

## Instruments


In [ ]:
import sys
sys.path.append("../drivers/")
from tn2211_drivers import *
import glob
import matplotlib.pyplot as plt
import math
import numpy as np
import time
from scipy.optimize import curve_fit
from scipy.signal import hilbert

In [ ]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

In [ ]:
scope = Scope("SDS")
gen = Generator("SDG")

## Level 1

### Step 1: Propagating wavefronts in a cable

In [ ]:
gen.write("*RST")
# Square pulse from 0 to 1V
gen.write("C1:BSWV WVTP,SQUARE,AMP,1,FRQ,20e3,OFST,0.5")
gen.write("C1:OUTP ON")
gen.write("C1:SYNC ON,TYPE,CH1")

# Reset all settings 
scope.write("*RST")
# Channel 1: the waveform
scope.write("CHAN1:SWIT ON")
scope.write("CHAN1:COUP DC")
scope.write("CHAN1:SCAL .2")
scope.write("CHAN1:OFFS -0.5")
# Channel 4: the trigger
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:COUP DC")
scope.write("CHAN4:SCAL .1")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")

scope.write("TIM:SCAL 10e-6")

# After we do a *RST, we need to let the oscillocope rest for a bit
# Otherwise, we start getting IO errors when reading data like a screenshot
# And have to power cycle it to get it working again. 
time.sleep(10)

In [ ]:
# A handy helper function
def get_trace_and_plot(title, v1, v2, t1, t2):
    t,v = scope.get_trace(1)
    plt.plot(t/1e-9,v)
    plt.xlabel("Time (ns)")
    plt.ylabel("Voltage at end of cable (V)")
    plt.title(title)
    
    # Zoom  in on your data
    #plt.xlim(,)
    #plt.ylim(,)
    
    # Add some vertical and horizontal cursors (adjust positions 
    for vx in v1,v2:
        plt.axhline(vx, ls=":", c="grey")
    for tx in t1,t2:
        plt.axvline(tx, ls=":", c="grey")

    plt.title(r"%s: $\Delta t$ = %.2f ns, $\Delta v$ = %.2f" % (title, (t2-t1),(v2-v1)))
    return t,v

Take some screenshots with different time axis zooms for your logbook:

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
# The cursors to plot
v1 = 0
v2 = v1 + 0.1
t1 = 0 
t2 = t1 + 100 # time in nanoseconds
plt.figure(figsize=(12,4), dpi=300)
t,v = get_trace_and_plot("add a title here", v1, v2, t1, t2)

### Step 2: Bouncing waves and RC times of cables

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
# The cursors to plot
v1 = 0
v2 = v1 + 0.1
t1 = 0 
t2 = t1 + 100 # time in nanoseconds
plt.figure(figsize=(12,6), dpi=300)
t,v = get_trace_and_plot("add a title here", v1, v2, t1, t2)

### Step 3: Probing both ends of the cable and terminating the cable

In [ ]:
scope.write("CHAN2:SWIT ON")
scope.write("CHAN2:COUP DC")
scope.write("CHAN2:SCAL .2")
scope.write("CHAN2:OFFS -0.5")

In [ ]:
# A handy helper function
def get_both_and_plot(title):
    scope.write("TRIG:STOP")
    t,v1 = scope.get_trace(1)
    t,v2 = scope.get_trace(2)
    scope.write("TRIG:RUN")
    plt.plot(t/1e-9,v1, label="End of cable")
    plt.plot(t/1e-9,v2, label="Start of cable")
    plt.legend()
    plt.xlabel("Time (ns)")
    plt.ylabel("Voltage (V)")
    plt.title(title)
    return t,v1,v2

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
plt.figure(figsize=(12,6), dpi=300)
t,v1,v2 = get_both_and_plot("add a title here")

Now terminate the scope end with the 50 ohm terminator:

In [ ]:
plt.figure(figsize=(12,6), dpi=300)
t,v3,v4 = get_both_and_plot("add a title here")

Compare terminated and non terminated ("high impedance"):

In [ ]:
plt.figure(figsize=(12,6), dpi=300)
plt.plot(t/1e-9,v1, label="High impedance")
plt.plot(t/1e-9,v3, label="50 ohm terminated")
plt.legend()
plt.xlabel("Time (ns)")
plt.ylabel("Voltage (V)")

### Step 4: Effect of “probing” on the rising edge of the pulse

In [ ]:
scope.write("CHAN2:SWIT OFF")
scope.write("CHAN1:SCAL 10e-3")
scope.write("TIM:SCAL 50e-9")
scope.write("TIM:DEL 200e-9")

In [ ]:
scope.get_screenshot()

Take two traces: one with the jumper wire connected that goes to Ch2 and one with the jumper wire removed:

In [ ]:
t,v1 = scope.get_trace(1)

In [ ]:
t,v2 = scope.get_trace(1)

In [ ]:
plt.figure(figsize=(12,6), dpi=300)
plt.plot(t/1e-9,v1, label="Cable connected to input for Ch2")
plt.plot(t/1e-9,v2, label="Cable to Ch2 disconnected")
plt.legend()
plt.xlabel("Time (ns)")
plt.ylabel("Voltage (V)")
plt.title("Cable to channel 2 = X meters")

Make a hypothesis of what is causing you are seeing, try repeating the experiment with a longer cable (2 meters) going to Ch2

### Step 5: Standing wave resonances of cables


Code to set up the equipment and to define some helper functions:

In [ ]:
# Set up the generators
gen.write("*RST")
time.sleep(2)
gen.write("C1:SWWV STATE,ON,SINE,AMP,1,TRSR,INT,TRMD,ON")
gen.write("C1:OUTP ON")
gen.write("C1:SYNC ON")

# We will use Ch4 of the scope connected to the sync out of the 
# generator for triggering
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:SCAL 2")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")
scope.write("TRIG:RUN")

# Configure the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN1:COUP AC")
scope.write("CHAN1:SCAL 1")
scope.write("CHAN1:OFFS 0")
scope.write("CHAN1:VIS ON")
scope.write("TIM:SCAL .1e-3")
scope.write("ACQ:MDEP 1M")

def setup_frequency_sweep(start_frequency, stop_frequency, sweep_time=1):
    # Ok, scope time base on screen goes in increments 1, 2, 5, 10...    
    mantissa = float(("%e" %  sweep_time).split("e")[0])
    exponent = float(("%e" %  sweep_time).split("e")[1])
    allowed = np.array([1, 2, 5, 10])
    idx = np.abs(allowed - mantissa).argmin()
    mantissa = allowed[idx]
    sweep_time = float("%fe%d" % (mantissa, exponent))
    print("Picking sweep time %e based on nearest allowed division of scope display" % sweep_time)
    
    # Configurting sweep, page 26 of manual of SDG1000 
    gen.write("C1:SWWV STATE,ON")
    gen.write("C1:SWWV TIME,%f" % sweep_time)
    gen.write("C1:SWWV STOP,%f" % stop_frequency)
    gen.write("C1:SWWV START,%f" % start_frequency)
    
    # This will automatically set up the time base of the scope in a good way :)
    scope.write("TIM:SCAL %f" % (sweep_time/10)) # The display has 10 "divisions"...
    scope.write("TIM:DEL %f" % (sweep_time/2)) # Set horizontal position to show full sweep on screen

def take_and_plot_sweep(title):
    scope.write("TRIG:STOP")
    t,v = scope.get_trace(1, npoints='all')
    scope.write("TRIG:RUN")
    
    f, A = make_amplitude_vs_freq(v, f1, f2)
    
    plt.figure(figsize=(12,4), dpi=300)
    plt.plot(f/1e6, A)
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Voltage amplitude")
    plt.title(title)
    return f,A

# We will just measure the output amplitude as a function of frequency and not divide by the input
def make_amplitude_vs_freq(v, f_start, f_stop, navg=10):
    # we will average to reduce the noise: we need a lot of points in time to measure 1 MHz but do not need 1 million 
    # points in our calculated frequency reponse function. 
    N = len(v) // navg
    f = np.linspace(f_start, f_stop, len(v)//navg)
    amp = np.abs(hilbert(v))
    amp = np.sqrt(np.average(np.reshape(amp[0:N*navg]**2,(N,navg)),axis=1))
    n_crop = len(amp)//500 # crop out some frequencies at the start and end due to FFT periodic boundary conditions
    print("Initial frequency range %.3f kHz to %.3f kHz" % (f[0]/1e3, f[-1]/1e3))
    print("Cropping from frequency %.3f kHz to %.3f kHz due to FFT" % (f[n_crop]/1e3, f[-n_crop]/1e3))
    return f[n_crop:-n_crop], np.abs(amp[n_crop:-n_crop])

f1 = 0.1e6
f2 = 30e6
setup_frequency_sweep(f1, f2, sweep_time = 0.1)

In [ ]:
scope.get_screenshot()

Code for running a frequency sweep and making a plot:

In [ ]:
f1 = 0.1e6  # You can change f1 and f2 to achieve different frequency ranges
f2 = 30e6

setup_frequency_sweep(f1, f2, sweep_time = 0.1)
f, A = take_and_plot_sweep("a title")

In [ ]:
# copy paste this cell for doing the different tests in level 2 and 3 too
f1 = 0.1e6  # You can change f1 and f2 to achieve different frequency ranges
f2 = 30e6

setup_frequency_sweep(f1, f2, sweep_time = 0.1)
f, A = take_and_plot_sweep("a title")